In [ ]:
from pathlib import Path
from ultralytics import YOLO
import shutil
import random
import cv2


base_dataset_path = Path(r'E:\sih\satriya_dataset\Sattriya-08 Double Handed Mudra’s Dataset')
prepared_dataset_path = Path(r'E:\sih\satriya_dataset\prepared_mudra_dataset')

# Clean previous prepared dataset (if any)
if prepared_dataset_path.exists():
    shutil.rmtree(prepared_dataset_path)
(prepared_dataset_path / 'images/train').mkdir(parents=True, exist_ok=True)
(prepared_dataset_path / 'images/val').mkdir(parents=True, exist_ok=True)
(prepared_dataset_path / 'labels/train').mkdir(parents=True, exist_ok=True)
(prepared_dataset_path / 'labels/val').mkdir(parents=True, exist_ok=True)

class_names = [f.name for f in base_dataset_path.iterdir() if f.is_dir()]
class_names.sort()
print(f"✅ Found Mudra Classes: {class_names}")

# Map each class to an integer ID
class_to_id = {name: idx for idx, name in enumerate(class_names)}


all_images = []
for mudra_folder in base_dataset_path.iterdir():
    if mudra_folder.is_dir():
        for view_folder in mudra_folder.iterdir():
            if view_folder.is_dir():
                for img_file in view_folder.glob("*.*"):
                    if img_file.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                        all_images.append((img_file, mudra_folder.name))

print(f"Total images found: {len(all_images)}")

random.shuffle(all_images)
split_idx = int(0.8 * len(all_images))
train_images = all_images[:split_idx]
val_images = all_images[split_idx:]

def copy_with_labels(image_list, split):
    """Copy images and create dummy YOLO labels for now."""
    for img_path, mudra_name in image_list:
        dest_img = prepared_dataset_path / f'images/{split}/{img_path.name}'
        shutil.copy(img_path, dest_img)

        # Dummy full-frame label for YOLO: (class_id x_center y_center width height)
        label_path = prepared_dataset_path / f'labels/{split}/{img_path.stem}.txt'
        with open(label_path, "w") as f:
            f.write(f"{class_to_id[mudra_name]} 0.5 0.5 1.0 1.0\n")

copy_with_labels(train_images, "train")
copy_with_labels(val_images, "val")

print("✅ YOLO dataset structure ready!")

data_yaml = f"""
train: {prepared_dataset_path.as_posix()}/images/train
val: {prepared_dataset_path.as_posix()}/images/val
nc: {len(class_names)}
names: {class_names}
"""
data_yaml_path = Path(r'E:\sih\satriya_dataset\mudra_data.yaml')
data_yaml_path.write_text(data_yaml)
print(f"✅ data.yaml created at {data_yaml_path}")


model = YOLO('yolov8s.pt')  # or yolov8m.pt / yolov8n.pt depending on resources

model.train(
    data=str(data_yaml_path),
    epochs=10,
    batch=8,
    imgsz=416,
    project='satriya_mudra_project',
    name='yolov8_mudra',
    cache=False,
    workers=0
)

# Save trained model
model_save_path = Path(r'E:\sih\satriya_dataset\yolov8_mudra.pt')
model.save(model_save_path)
print(f"✅ Trained model saved at: {model_save_path}")


✅ Found Mudra Classes: ['Bardhaman', 'Dol', 'Gajadanta', 'Kaput', 'Karkat', 'Makar1', 'Makar2', 'Samput']
Total images found: 2305
✅ YOLO dataset structure ready!
✅ data.yaml created at E:\sih\satriya_dataset\mudra_data.yaml
New https://pypi.org/project/ultralytics/8.3.213 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.207  Python-3.12.10 torch-2.8.0+cpu CPU (12th Gen Intel Core(TM) i5-12450H)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\sih\satriya_dataset\mudra_data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=4

In [9]:
from ultralytics import YOLO
import cv2
from pathlib import Path
import random

# ----------------------------------------------------------
# 1️⃣ Paths and Model
# ----------------------------------------------------------
model_path = Path(r'E:\sih\satriya_dataset\yolov8_mudra.pt')
data_path = Path(r'E:\sih\satriya_dataset\prepared_mudra_dataset\images\val')

# Load trained model
model = YOLO(model_path)

# Class names (update if more exist)
class_names = [
    'Bardhaman', 'Dol', 'Gajadanta', 'Hamsapaksha', 'Kataka',
    'Kartari', 'Pataka', 'Pushpaputa', 'Tripataka', 'Shikhara'
]

# ----------------------------------------------------------
# 2️⃣ Pick a random image for testing
# ----------------------------------------------------------
test_images = list(data_path.glob("*.jpg")) + list(data_path.glob("*.png")) + list(data_path.glob("*.jpeg"))

if not test_images:
    print("❌ No images found in:", data_path)
    exit()

test_image_path = random.choice(test_images)
print(f"🧪 Testing on image: {test_image_path}")

# ----------------------------------------------------------
# 3️⃣ Load and check image
# ----------------------------------------------------------
image = cv2.imread(str(test_image_path))
if image is None:
    print("❌ Failed to read image. Please check path or format.")
    exit()

# ----------------------------------------------------------
# 4️⃣ Run YOLO inference
# ----------------------------------------------------------
results = model(str(test_image_path))

if len(results[0].boxes) == 0:
    print("⚠️ No mudra detected.")
else:
    for detection in results[0].boxes:
        x1, y1, x2, y2 = detection.xyxy[0].tolist()
        conf = detection.conf[0].item()
        cls_id = int(detection.cls[0].item())
        conf_percent = conf * 100
        mudra_name = class_names[cls_id] if cls_id < len(class_names) else "Unknown"

        label = f"{mudra_name}: {conf_percent:.1f}%"

        # Draw bounding box + background rectangle for text
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.rectangle(image, (int(x1), int(y1) - 25), (int(x1) + 220, int(y1)), (0, 255, 0), -1)
        cv2.putText(image, label, (int(x1) + 5, int(y1) - 7),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

        print(f"✅ Detected {mudra_name} ({conf_percent:.1f}%) at [{int(x1)}, {int(y1)}, {int(x2)}, {int(y2)}]")

# ----------------------------------------------------------
# 5️⃣ Show detection result
# ----------------------------------------------------------
# Resize image to fit the screen if it’s too large
screen_res = (500, 720)
scale_width = screen_res[0] / image.shape[1]
scale_height = screen_res[1] / image.shape[0]
scale = min(scale_width, scale_height)
window_width = int(image.shape[1] * scale)
window_height = int(image.shape[0] * scale)

cv2.namedWindow("Mudra Detection", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Mudra Detection", window_width, window_height)
cv2.imshow("Mudra Detection", image)

print("\n👁️ Press any key on the image window to close it...")
cv2.waitKey(0)
cv2.destroyAllWindows()


🧪 Testing on image: E:\sih\satriya_dataset\prepared_mudra_dataset\images\val\unscreen-143 (2).png

image 1/1 E:\sih\satriya_dataset\prepared_mudra_dataset\images\val\unscreen-143 (2).png: 256x416 1 Samput, 72.2ms
Speed: 0.9ms preprocess, 72.2ms inference, 0.9ms postprocess per image at shape (1, 3, 256, 416)
✅ Detected Pushpaputa (99.6%) at [0, 2, 640, 359]

👁️ Press any key on the image window to close it...
